In [ ]:
#

In [ ]:
#

In [ ]:
!gcloud auth login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=BeMEfLzczTMt7TTidknILAqL4Hf0AB&prompt=consent&token_usage=remote&access_type=offline&code_challenge=Wm5WtfflmFrk-LnoXHo2h3Cz8H9-YpiGuyKwcQAiWCU&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AXlqoi5bNwUUpPh3plln4qYkD2mwYlTnjxd9uQY5mSBWgqQd6gUO6okYh-5-GxhtBW580g

You are now logged in as [ghurbal.akbar@gmail.com].
Your current projec

In [ ]:
from google.colab import userdata

# Private values live here in the notebook, like the API keys — never in the repo.
GCP_DATASET_PATH = "gs://akbar-december-2024-backup/OSTRIS_Arabic_Suno_Finetuning/dataset"
GCP_BACKUP_BASE = "gs://akbar-december-2024-backup/OSTRIS_Arabic_Suno_Finetuning"

# pron_dataset is only for the AR-only pron run; NOT needed for quran_long_aya_r8.
# GCP_PRON_DATASET_PATH = "gs://akbar-december-2024-backup/OSTRIS_Arabic_Suno_Finetuning/pron_dataset"

# The long-aya run's dataset (~28 GiB) — pulled in PARALLEL during launch (next cell).
GCP_QURAN_LONG_DATASET_PATH = "gs://akbar-december-2024-backup/OSTRIS_Arabic_Suno_Finetuning/quran_long_aya_dataset"

with open('/root/.secrets.env', 'w') as f:
    f.write(f'export DEEPSEEK_API_KEY="{userdata.get("DEEPSEEK_API_KEY")}"\n')
    f.write(f'export HF_TOKEN="{userdata.get("HF_TOKEN")}"\n')
    f.write(f'export GCP_DATASET_PATH="{GCP_DATASET_PATH}"\n')
    # f.write(f'export GCP_PRON_DATASET_PATH="{GCP_PRON_DATASET_PATH}"\n')
    f.write(f'export GCP_QURAN_LONG_DATASET_PATH="{GCP_QURAN_LONG_DATASET_PATH}"\n')
    f.write(f'export GCP_BACKUP_BASE="{GCP_BACKUP_BASE}"\n')

marker = 'source /root/.secrets.env'
with open('/root/.bashrc') as f:
    already_wired = marker in f.read()
if not already_wired:
    with open('/root/.bashrc', 'a') as f:
        f.write(f'\n{marker}\n')

print("Secrets staged. Open the Colab terminal now.")


Secrets staged. Open the Colab terminal now.


In [ ]:
# opencode global default model + reasoning variant.
# Global (~/.config/opencode) => applies to ALL projects on this VM. It is wiped on a
# fresh VM, so this cell must run every session (same as the bucket names above).
OPENCODE_MODEL   = "deepseek/deepseek-flash"   # <provider>/<model>; bump when DeepSeek ships a new Flash
OPENCODE_VARIANT = "high"                      # reasoning effort: low | high | max

import json, os

os.makedirs("/root/.config/opencode", exist_ok=True)
opencode_cfg = {
    "$schema": "https://opencode.ai/config.json",
    "model": OPENCODE_MODEL,
    "agent": {
        "build": {"model": OPENCODE_MODEL, "variant": OPENCODE_VARIANT},
        "plan":  {"model": OPENCODE_MODEL, "variant": OPENCODE_VARIANT},
    },
}
# Write both spellings so whichever opencode loads (json or jsonc) carries the setting.
for _p in ("/root/.config/opencode/opencode.json", "/root/.config/opencode/opencode.jsonc"):
    with open(_p, "w") as _f:
        json.dump(opencode_cfg, _f, indent=2)

print(f"opencode default: {OPENCODE_MODEL} [{OPENCODE_VARIANT}] — restart opencode to apply")


opencode default: deepseek/deepseek-flash [high] — restart opencode to apply


In [ ]:
!curl -Lk 'https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64' --output vscode_cli.tar.gz
!tar -xf vscode_cli.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   162  100   162    0     0    101      0  0:00:01  0:00:01 --:--:--   101
100 12.3M  100 12.3M    0     0  7339k      0  0:00:01  0:00:01 --:--:-- 7339k


## Training setup (fresh VM)

```bash
# 0. Load the staged env (bucket paths). The cell above wrote /root/.secrets.env.
. /root/.secrets.env

# 1. The redirect target must exist BEFORE bash opens it. /content/logs
#    does not exist on a fresh Colab VM, and a failed redirect kills the
#    whole line silently (setup.sh never starts).
mkdir -p /content/logs

# 2. Clone (skip if already present, so re-pasting the block is safe).
[ -d maqamrock-yue2-lora-finetuning ] || git clone https://github.com/akbargherbal/maqamrock-yue2-lora-finetuning.git
cd maqamrock-yue2-lora-finetuning
# 2b. The long-aya run's config + docs live on THIS branch (not pron-lora-ar-only).
git checkout pron-lora-long

# 3. Launch setup DETACHED. A bare '&' job is killed by SIGHUP when this
#    terminal closes; setsid + nohup + disown survive the tunnel handoff.
#    Training mode also pulls the old v2 dataset into /content/yue2_dataset — ignore that.
setsid nohup bash bootstrap/setup.sh > /content/logs/setup.log 2>&1 < /dev/null &
disown

# 3b. Pull the quran_long_aya dataset (~28 GiB) IN PARALLEL — it downloads while you
#     authenticate the tunnel below, so it is ready when setup finishes. rsync is
#     incremental, so re-pasting is cheap. Log: /content/logs/quran_ds.log
setsid nohup gsutil -m rsync -r "$GCP_QURAN_LONG_DATASET_PATH" /content/quran_long_aya_dataset \
  > /content/logs/quran_ds.log 2>&1 < /dev/null &
disown

# 4. Confirm both actually started (should print PIDs).
pgrep -af 'bootstrap/setup.sh'; pgrep -af 'quran_long_aya_dataset'

# 5. Authenticate the tunnel in the FOREGROUND; keep this terminal open.
/content/code tunnel
```

When setup **and** the dataset rsync finish, verify the counts (81,006 train `.mp3` + `.txt`,
352/352 val, 14/14 smoke), then start the run per `docs/PRON_LORA_LONG.md`.


## Inference setup (fresh VM)

```bash
# 1. The redirect target must exist BEFORE bash opens it. /content/logs
#    does not exist on a fresh Colab VM, and a failed redirect kills the
#    whole line silently (setup.sh never starts).
mkdir -p /content/logs

# 2. Clone (skip if already present, so re-pasting the block is safe).
[ -d maqamrock-yue2-lora-finetuning ] || git clone https://github.com/akbargherbal/maqamrock-yue2-lora-finetuning.git
cd maqamrock-yue2-lora-finetuning

# 3. Launch INFERENCE setup DETACHED (--inference clones audio.cpp and
#    pre-warms the bf16 GGUF + VAE; it skips the dataset and ai-toolkit).
setsid nohup bash bootstrap/setup.sh --inference > /content/logs/setup.log 2>&1 < /dev/null &
disown

# 4. Confirm it actually started before moving on (should print a PID).
pgrep -af 'bootstrap/setup.sh'

# 5. Authenticate the tunnel in the FOREGROUND; keep this terminal open.
/content/code tunnel
```

After setup finishes, build audio.cpp once (it is only cloned, never built)
and pull the converted LoRA + runner scripts from the run's GCS prefix.


```js
function ClickConnect() {
  console.log("Checking connection status...");
  const colabButton = document.querySelector("colab-connect-button");
  if (colabButton && colabButton.shadowRoot) {
    const connectBtn = colabButton.shadowRoot.querySelector("#connect");
    if (connectBtn) {
      connectBtn.click();
      console.log("Connect button clicked ✅");
    } else {
      console.log("Connect button not found ❌");
    }
  } else {
    console.log("colab-connect-button element not found ❌");
  }
}
setInterval(ClickConnect, 60000); // Clicks every 60 seconds

```

In [ ]:
%%bash
curl -fsSL https://opencode.ai/v2/install | bash &



Installing opencode version: 2.0.16
Successfully added opencode to $PATH in /root/.bashrc

                                 ▄     
█▀▀█ █▀▀█ █▀▀█ █▀▀▄ █▀▀▀ █▀▀█ █▀▀█ █▀▀█
█░░█ █░░█ █▀▀▀ █░░█ █░░░ █░░█ █░░█ █▀▀▀
▀▀▀▀ █▀▀▀ ▀▀▀▀ ▀  ▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀


OpenCode includes free models, to start:

cd <project>  # Open directory
opencode      # Run command

For more information visit https://opencode.ai/v2/docs




######################################################################## 100.0%


```bash
mkdir -p /content/logs
cd /content
. ~/.bashrc
# ai-toolkit SOURCE only — the agent needs to read toolkit/lora_special.py,
# not run training. Skip its requirements.txt entirely (that's the CUDA pin).
git clone https://github.com/ostris/ai-toolkit.git &

# This project's repo
[ -d maqamrock-yue2-lora-finetuning ] || git clone https://github.com/akbargherbal/maqamrock-yue2-lora-finetuning.git
cd maqamrock-yue2-lora-finetuning
git checkout pron-lora-ar-only

# CPU-only torch + safetensors — all the merge tool actually needs to run
pip install -q --no-input torch --index-url https://download.pytorch.org/whl/cpu
pip install -q --no-input safetensors

# The existing converter script + its current outputs (for the invariant sha256 test),
# ~133 MiB, not the multi-GB inference bundle
mkdir -p /content/converter/out
gsutil -m rsync -r "$GCP_BACKUP_BASE/audiocpp_inference/converter" /content/ &converter/out

/content/code tunnel
```

In [ ]:
1

1